# 3. Example training from scratch (optional)
**Before you start:** select the **Python (DECODE Intel)** kernel. Run each cell from top to bottom (Shift+Enter). If you change `config.yaml`, rerun the first cell to reload it. Nothing is saved unless a cell tells you it will save.
**Goal:** prepare DECODE to learn from its own simulated example. This does not use your movie and does not start training automatically.

On this Mac DECODE runs on the CPU; training may be slow. The default is two epochs: epoch 0 is a baseline evaluation and epoch 1 performs training. This checks the workflow, not scientific accuracy.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import yaml

# Start Jupyter from the project folder or decode_walkthrough/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'decode_walkthrough' / 'config.yaml').is_file():
    ROOT = ROOT.parent
config_path = ROOT / 'decode_walkthrough' / 'config.yaml'
if not config_path.is_file():
    raise FileNotFoundError('Start Jupyter in the project folder or decode_walkthrough/.')
with config_path.open() as f:
    CFG = yaml.safe_load(f)
def project_path(value):
    if value is None: return None
    p = Path(value).expanduser()
    return p if p.is_absolute() else ROOT / p
EXAMPLE = project_path(CFG['paths']['example_folder'])
WORK = project_path(CFG['paths']['work_folder'])
print('Project:', ROOT, '| Config:', config_path, '| Example exists:', EXAMPLE.exists())


## A. Load example settings
The PSF, camera and network settings come from the bundled `param_run.yaml`. The small run settings come from `config.yaml`. Printing a device does not confirm a complete training run will succeed.

In [ ]:
import torch
import decode.utils.param_io
print('CUDA:', torch.cuda.is_available())
param = decode.utils.param_io.load_params(str(project_path(CFG['paths']['example_parameters'])))
param.InOut.calibration_file = str(project_path(CFG['paths']['example_calibration']).resolve())
param.InOut.checkpoint_init = None
train_cfg = CFG['example_training']
param.Hardware.device = ('cuda:0' if torch.cuda.is_available() else 'cpu') if train_cfg['device'] == 'auto' else train_cfg['device']
param.Hardware.device_simulation = param.Hardware.device
param.Hardware.num_worker_train = train_cfg['workers']
param.HyperParameter.batch_size = train_cfg['batch_size']
param.HyperParameter.epochs = train_cfg['epochs']  # demonstration only
param.HyperParameter.pseudo_ds_size = train_cfg['training_samples']
param.TestSet.test_size = train_cfg['test_samples']
# StepLR avoids conflicting scheduler defaults when DECODE loads the example YAML.
param.HyperParameter.learning_rate_scheduler = 'StepLR'
param.HyperParameter.learning_rate_scheduler_param = decode.utils.param_io.RecursiveNamespace(
    step_size=train_cfg['scheduler_step_size'], gamma=train_cfg['scheduler_gamma'])
print('Device:', param.Hardware.device)


## B. Create a training configuration
When ready, set `example_training.write_config: true` in `config.yaml` and rerun the first cell and this cell. It creates a new YAML file and prints a command. It refuses to overwrite a previous configuration; rename the old file if you want another run.


In [ ]:
WRITE_CONFIG = CFG['example_training']['write_config']
if WRITE_CONFIG:
    WORK.mkdir(parents=True, exist_ok=True)
    (WORK / 'scratch_runs').mkdir(exist_ok=True)
    param.InOut.experiment_out = str((WORK / 'scratch_runs').resolve())
    config_path = WORK / 'scratch_params.yaml'
    if config_path.exists(): raise FileExistsError(f'Choose a fresh path: {config_path}')
    decode.utils.param_io.save_params(str(config_path), param)
    print('Config saved:', config_path)
    print('In a terminal with the DECODE environment activated, run:')
    print(f'python -m decode.neuralfitter.train.live_engine -p "{config_path}"')
else:
    print('Set example_training.write_config: true in config.yaml to save a new config and display the command.')


## C. Run in Terminal
Activate `decode_intel`, paste the printed command, and wait for the run to finish. DECODE creates a run folder under `decode_work/scratch_runs`; keep its `model_*.pt` and `param_run.yaml` together. Training can be interrupted in Terminal with Ctrl+C.

**To analyse your own movie:** you need suitable channel-specific calibration, camera and simulated-emitter parameters. The own-data entries in the walkthrough config do not automatically build that DECODE training configuration.
